In [1]:
import importlib.util, json, math, random, re, subprocess, sys
from importlib.metadata import version
from packaging.version import Version
from pathlib import Path
_REQUIRED = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "networkx": "networkx", "sklearn": "scikit-learn", "sentence_transformers": "sentence-transformers", "torch": "torch", "accelerate": "accelerate"}
_missing = [package for module, package in _REQUIRED.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
if Version(version("accelerate")) < Version("1.1.0"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "accelerate>=1.1.0"])
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find shared config.json from the notebook working directory")
CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
SEED = int(CONFIG["random_seed"])
random.seed(SEED)
np.random.seed(SEED)
for key in ("data_dir", "artifact_dir", "results_dir"):
    (ROOT / CONFIG[key]).mkdir(parents=True, exist_ok=True)
ARTIFACTS = ROOT / CONFIG["artifact_dir"]
RESULTS = ROOT / CONFIG["results_dir"]
print(f"Setup complete; smoke_test_mode={CONFIG['smoke_test_mode']}; seed={SEED}")

Setup complete; smoke_test_mode=True; seed=42


# 04 — Retrieval, fusion, and reranking smoke path

Notebook 03 fine-tunes the cached bi-encoder from profile-disjoint synthetic MNRL triplets. This notebook compares the base and fine-tuned encoders, tunes retrieval weights on validation profiles, and evaluates only on held-out profiles. All labels/results are `SYNTHETIC_SMOKE_TEST`, not evidence of real relevance or KG benefit; cross-encoder scores are ranking scores, not probabilities.

In [2]:
import hashlib
import itertools
import json
import math
import re
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from tools.policy_eligibility import evaluate_eligibility

import numpy as np
import pandas as pd
import networkx as nx
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

policies_df = pd.read_parquet(ARTIFACTS / "policies.parquet")
queries_df = pd.read_csv(ARTIFACTS / "queries.csv")
intents = {row["query_id"]: row for row in json.loads((ARTIFACTS / "intents.json").read_text(encoding="utf-8"))}
kg = nx.read_graphml(ARTIFACTS / "kg.graphml")
finetune_status = json.loads((RESULTS / "finetune.json").read_text(encoding="utf-8"))
label_status = json.loads((RESULTS / "data_status.json").read_text(encoding="utf-8"))
splits = json.loads((ARTIFACTS / "splits.json").read_text(encoding="utf-8"))
synthetic_labels = pd.read_parquet(ARTIFACTS / "synthetic_smoke_labels.parquet")

if finetune_status["status"] != "COMPLETED_SYNTHETIC_SMOKE_TEST":
    raise RuntimeError("Notebook 03 must finish synthetic MNRL training before notebook 04")
if label_status["human_graded_rows"] != 0 or not synthetic_labels["annotator"].eq("SYNTHETIC_SMOKE_TEST").all():
    raise ValueError("Notebook 04 accepts only synthetic smoke-test labels")
if not synthetic_labels["provenance"].eq("SYNTHETIC").all():
    raise ValueError("Synthetic label provenance must be SYNTHETIC")

policy_records = policies_df.to_dict(orient="records")
policy_by_id = {str(row["policy_id"]): row for row in policy_records}
policy_ids = sorted(policy_by_id)
documents = {
    pid: " ".join(
        str(policy_by_id[pid][key])
        for key in ("policy_name", "category", "financial_goal", "coverage_type", "description")
        if pd.notna(policy_by_id[pid].get(key)) and policy_by_id[pid].get(key) != "" and (key != "category" or bool(policy_by_id[pid].get("category_verified")))
    )
    for pid in policy_ids
}
query_by_id = dict(zip(queries_df["query_id"], queries_df["raw_query"]))
query_ids = queries_df["query_id"].tolist()
query_texts = [query_by_id[qid] for qid in query_ids]
policy_texts = [documents[pid] for pid in policy_ids]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), norm="l2")
document_matrix = vectorizer.fit_transform(policy_texts)
try:
    import faiss
    HAS_FAISS = True
except (ImportError, OSError):
    HAS_FAISS = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE_ALPHA = 0.5
SMOKE_BETA = 0.5

def backend_plan(faiss_available, encoder_available, network_allowed, cross_encoder_loaded):
    return {
        "vector": "faiss_exact_ip" if faiss_available else "numpy_exact",
        "encoder": "pretrained_biencoder" if encoder_available and network_allowed else "tfidf_fallback",
        "cross_encoder": "cross_encoder_ranking_score" if cross_encoder_loaded else "dense_ranking_score_fallback",
        "network": "allowed" if network_allowed else "disabled_local_only",
    }

def minmax(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    lo, hi = float(values.min()), float(values.max())
    return np.full_like(values, 0.5) if math.isclose(lo, hi, abs_tol=1e-12) else (values - lo) / (hi - lo)

def tokens(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def bm25_scores(query, ids):
    docs = [tokens(documents[pid]) for pid in ids]
    query_terms = tokens(query)
    n_docs = len(docs)
    if not n_docs:
        return np.array([], dtype=float)
    lengths = [len(doc) for doc in docs]
    avg_len = sum(lengths) / n_docs or 1.0
    doc_freq = {term: sum(term in set(doc) for doc in docs) for term in set(query_terms)}
    scores = []
    for doc, length in zip(docs, lengths):
        score = 0.0
        for term in query_terms:
            tf = doc.count(term)
            if tf:
                idf = math.log(1 + (n_docs - doc_freq[term] + 0.5) / (doc_freq[term] + 0.5))
                score += idf * tf * 2.2 / (tf + 1.2 * (0.25 + 0.75 * length / avg_len))
        scores.append(score)
    return np.asarray(scores, dtype=float)

def dense_scores(query, ids, use_faiss=None):
    if not ids:
        return np.array([], dtype=float)
    positions = [policy_ids.index(pid) for pid in ids]
    matrix = document_matrix[positions].toarray().astype("float32")
    query_vector = vectorizer.transform([query]).toarray().astype("float32")
    return (matrix @ query_vector[0]).astype(float)

def eligibility_status(policy, intent):
    return evaluate_eligibility(policy, intent)["status"]

def kg_features(policy, intent):
    return {
        "goal": bool(intent.get("all_goals")) and policy.get("financial_goal") in intent["all_goals"],
        "risk": bool(intent.get("risk_tolerance")) and policy.get("risk_tolerance_inferred") == intent["risk_tolerance"],
        "premium": bool(intent.get("premium_level")) and policy.get("premium_level_inferred") == intent["premium_level"],
    }

def kg_preference(policy, intent, weights):
    return sum(weights[name] * float(value) for name, value in kg_features(policy, intent).items())

def ndcg_at_4(ranked_ids, query_id, eligible_ids):
    grade_by_policy = label_by_query.get(query_id, {})
    dcg = sum((2 ** grade_by_policy.get(pid, 0) - 1) / math.log2(rank + 1) for rank, pid in enumerate(ranked_ids[:4], 1))
    ideal = sorted((grade_by_policy.get(pid, 0) for pid in eligible_ids), reverse=True)[:4]
    idcg = sum((2 ** grade - 1) / math.log2(rank + 1) for rank, grade in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0

# Cache embeddings so notebook reruns do not reload/encode unchanged artifacts.
model_paths = {
    "baseline": CONFIG["embedding_model_name"],
    "fine_tuned": str(ROOT / finetune_status["experiments"]["group_split"]["path"]),
}
embedding_signature = hashlib.sha256(
    json.dumps(
        {
            "models": model_paths,
            "training_sha256": finetune_status["experiments"]["group_split"]["training_sha256"],
            "queries": query_texts,
            "policies": policy_texts,
        },
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()
embedding_path = ARTIFACTS / "retrieval_embeddings_synthetic.npz"
embedding_meta_path = ARTIFACTS / "retrieval_embeddings_synthetic.json"
if embedding_path.is_file() and embedding_meta_path.is_file() and json.loads(embedding_meta_path.read_text(encoding="utf-8")).get("signature") == embedding_signature:
    with np.load(embedding_path, allow_pickle=False) as cache:
        embedding_arrays = {key: cache[key] for key in ("baseline_queries", "baseline_policies", "fine_tuned_queries", "fine_tuned_policies")}
else:
    embedding_arrays = {}
    for model_key, model_path in model_paths.items():
        model = SentenceTransformer(model_path, device=DEVICE)
        embedding_arrays[f"{model_key}_queries"] = model.encode(query_texts, batch_size=16, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        embedding_arrays[f"{model_key}_policies"] = model.encode(policy_texts, batch_size=16, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        del model
    np.savez_compressed(embedding_path, **embedding_arrays)
    embedding_meta_path.write_text(json.dumps({"provenance": "SYNTHETIC_SMOKE_TEST", "signature": embedding_signature, "models": model_paths}, indent=2), encoding="utf-8")
query_position = {qid: index for index, qid in enumerate(query_ids)}
policy_position = {pid: index for index, pid in enumerate(policy_ids)}
semantic_vectors = {
    model_key: {
        "queries": {qid: embedding_arrays[f"{model_key}_queries"][index] for index, qid in enumerate(query_ids)},
        "policies": {pid: embedding_arrays[f"{model_key}_policies"][index] for index, pid in enumerate(policy_ids)},
    }
    for model_key in ("baseline", "fine_tuned")
}

def transformer_dense_scores(query_id, ids, model_key):
    query_vector = semantic_vectors[model_key]["queries"][query_id]
    policy_matrix = np.asarray([semantic_vectors[model_key]["policies"][pid] for pid in ids])
    return policy_matrix @ query_vector

# Cache cross-encoder ranking scores for every eligible query-policy pair.
query_intents = {qid: intents[qid]["intent"] for qid in query_ids}
eligible_pairs = []
for query_id in query_ids:
    eligible_ids = [pid for pid in policy_ids if eligibility_status(policy_by_id[pid], query_intents[query_id]) == "eligible"]
    eligible_pairs.extend((query_id, pid) for pid in eligible_ids)
cross_signature = hashlib.sha256(json.dumps([CONFIG["cross_encoder_model_name"], eligible_pairs], sort_keys=True).encode("utf-8")).hexdigest()
cross_cache_path = ARTIFACTS / "cross_encoder_scores_synthetic.json"
cross_encoder = None
cross_encoder_error = None
if CONFIG["allow_model_downloads"]:
    try:
        cross_encoder = CrossEncoder(CONFIG["cross_encoder_model_name"], device=DEVICE)
    except (ImportError, OSError, RuntimeError, TimeoutError, ConnectionError) as exc:
        cross_encoder_error = f"{type(exc).__name__}: {exc}"

if cross_encoder is not None and cross_cache_path.is_file():
    cached_cross = json.loads(cross_cache_path.read_text(encoding="utf-8"))
else:
    cached_cross = {}
if cross_encoder is not None and cached_cross.get("signature") == cross_signature:
    cross_scores = cached_cross["scores"]
elif cross_encoder is not None:
    pairs = [[query_by_id[qid], documents[pid]] for qid, pid in eligible_pairs]
    raw_scores = cross_encoder.predict(pairs, batch_size=16, show_progress_bar=False, convert_to_numpy=True) if pairs else np.array([], dtype=float)
    cross_scores = {}
    for (query_id, policy_id), score in zip(eligible_pairs, np.asarray(raw_scores).reshape(-1)):
        cross_scores.setdefault(query_id, {})[policy_id] = float(score)
    cross_cache_path.write_text(json.dumps({"provenance": "SYNTHETIC_SMOKE_TEST", "signature": cross_signature, "scores": cross_scores}, indent=2), encoding="utf-8")
else:
    cross_scores = {}

BACKENDS = backend_plan(HAS_FAISS, True, bool(CONFIG["allow_model_downloads"]), cross_encoder is not None)
label_by_query = {
    query_id: dict(zip(rows["policy_id"], rows["grade"].astype(int)))
    for query_id, rows in synthetic_labels.groupby("query_id", sort=False)
}

kg_grid = [
    {"goal": goal / 4, "risk": risk / 4, "premium": premium / 4}
    for goal in range(5)
    for risk in range(5 - goal)
    for premium in [4 - goal - risk]
]

def rank_order(ids, scores, dense_scores, bm25_scores, cross_scores):
    return sorted(
        range(len(ids)),
        key=lambda index: (
            -float(scores[index]),
            -float(dense_scores[index]),
            -float(bm25_scores[index]),
            -float(cross_scores[index]),
            ids[index],
        ),
    )


def rank_tie_metrics(scores):
    scores = [float(score) for score in scores]
    tied_pairs = sum(math.isclose(left, right, rel_tol=0.0, abs_tol=1e-12) for left, right in zip(scores, scores[1:]))
    pair_count = max(0, len(scores) - 1)
    all_equal = len(scores) > 1 and tied_pairs == pair_count
    return {"tied_rank_fraction": tied_pairs / pair_count if pair_count else 0.0, "all_equal_final_scores": all_equal}


def summarize_rank_ties(results):
    scored_lists = [row.get("ranked_scores", [record["final_ranking_score"] for record in row.get("recommendations", [])]) for row in results]
    scored_lists = [scores for scores in scored_lists if len(scores) > 1]
    metrics = [rank_tie_metrics(scores) for scores in scored_lists]
    pair_count = sum(max(0, len(scores) - 1) for scores in scored_lists)
    tied_pairs = sum(round(metric["tied_rank_fraction"] * (len(scores) - 1)) for scores, metric in zip(scored_lists, metrics))
    all_equal_count = sum(metric["all_equal_final_scores"] for metric in metrics)
    return {
        "tied_rank_fraction": tied_pairs / pair_count if pair_count else 0.0,
        "all_equal_score_list_count": int(all_equal_count),
        "scored_list_count": len(scored_lists),
        "all_equal_list_fraction": all_equal_count / len(scored_lists) if scored_lists else 0.0,
    }


def assert_tie_collapse_limit(results):
    tie_metrics = summarize_rank_ties(results)
    if tie_metrics["all_equal_list_fraction"] > 0.2:
        raise RuntimeError("More than 20% of returned pipeline lists have all-equal final scores")
    return tie_metrics


def rank_query(query, intent, alpha=SMOKE_ALPHA, beta=SMOKE_BETA, kg_weights=None, model_key="baseline", query_id=None, ranker="pipeline"):
    if kg_weights is None:
        kg_weights = {"goal": 1 / 3, "risk": 1 / 3, "premium": 1 / 3}
    states = {pid: eligibility_status(policy_by_id[pid], intent) for pid in policy_ids}
    eligible = [pid for pid in policy_ids if states[pid] == "eligible"]
    if not eligible:
        return {"status": "NO_ELIGIBLE_POLICIES", "recommendations": [], "ranked_policy_ids": [], "ranked_scores": [], "eligible_candidate_count": 0, "display_cap": int(CONFIG["display_recommendations"]), "display_omitted_count": 0, "eligibility": states, "backend": BACKENDS, **rank_tie_metrics([])}

    dense = minmax(transformer_dense_scores(query_id, eligible, model_key) if query_id in query_position else dense_scores(query, eligible, use_faiss=False))
    bm25_raw = bm25_scores(query, eligible)
    bm25 = minmax(bm25_raw)
    if ranker == "id_order":
        candidate_ids = sorted(eligible)
        candidate_dense = np.zeros(len(candidate_ids), dtype=float)
        candidate_bm25 = np.zeros(len(candidate_ids), dtype=float)
        ce_norm = np.zeros(len(candidate_ids), dtype=float)
        final_scores = np.asarray([len(candidate_ids) - index for index in range(len(candidate_ids))], dtype=float)
        ce_kind = "id_order_baseline"
    elif ranker == "bm25_only":
        candidate_ids = eligible
        candidate_dense = np.zeros(len(candidate_ids), dtype=float)
        candidate_bm25 = bm25
        ce_norm = np.zeros(len(candidate_ids), dtype=float)
        final_scores = bm25
        ce_kind = "bm25_only_baseline"
    elif ranker == "pipeline":
        kg_norm = minmax([kg_preference(policy_by_id[pid], intent, kg_weights) for pid in eligible])
        bm25_kg = minmax((bm25 + kg_norm) / 2)
        candidate_scores = alpha * dense + (1 - alpha) * bm25_kg
        candidate_order = sorted(
            range(len(eligible)),
            key=lambda index: (-candidate_scores[index], -dense[index], -bm25[index], eligible[index]),
        )
        candidate_ids = [eligible[index] for index in candidate_order]
        candidate_dense = np.asarray([dense[index] for index in candidate_order], dtype=float)
        candidate_bm25 = np.asarray([bm25[index] for index in candidate_order], dtype=float)
        candidate_kg = np.asarray([kg_norm[index] for index in candidate_order], dtype=float)
        if query_id is not None and query_id in cross_scores:
            ce_values = [cross_scores[query_id].get(pid, float("nan")) for pid in candidate_ids]
            if all(np.isfinite(ce_values)):
                ce_norm = minmax(ce_values)
                ce_kind = "cross_encoder_ranking_score_not_probability"
            else:
                ce_norm = minmax(candidate_dense)
                ce_kind = "dense_ranking_score_fallback_not_probability"
        else:
            ce_norm = minmax(candidate_dense)
            ce_kind = "dense_ranking_score_fallback_not_probability"
        final_scores = beta * ce_norm + (1 - beta) * candidate_kg
    else:
        raise ValueError(f"Unknown ranker: {ranker}")

    order = rank_order(candidate_ids, final_scores, candidate_dense, candidate_bm25, ce_norm)
    ranked_policy_ids = [candidate_ids[index] for index in order]
    ranked_scores = [float(final_scores[index]) for index in order]
    display_cap = int(CONFIG["display_recommendations"])
    display_order = order[:display_cap]
    recommendations = [
        {
            "rank": rank,
            "policy_id": candidate_ids[index],
            "final_ranking_score": float(final_scores[index]),
            "cross_encoder_ranking_score": cross_scores.get(query_id, {}).get(candidate_ids[index]) if query_id is not None and ranker == "pipeline" else None,
            "cross_encoder_fallback_ranking_score": float(ce_norm[index]) if ranker == "pipeline" and ce_kind.startswith("dense") else None,
            "score_kind": "ranking_score_not_probability",
            "eligibility": states[candidate_ids[index]],
        }
        for rank, index in enumerate(display_order, 1)
    ]
    return {
        "status": "SUCCESS",
        "recommendations": recommendations,
        "ranked_policy_ids": ranked_policy_ids,
        "ranked_scores": ranked_scores,
        "eligible_candidate_count": len(eligible),
        "display_cap": display_cap,
        "display_omitted_count": max(0, len(eligible) - len(recommendations)),
        "eligibility": states,
        "backend": BACKENDS,
        **rank_tie_metrics(final_scores[order]),
    }
def query_metrics(query_id, model_key, alpha, beta, kg_weights, ranker="pipeline"):
    query = query_by_id[query_id]
    intent = query_intents[query_id]
    result = rank_query(query, intent, alpha, beta, kg_weights, model_key, query_id, ranker)
    eligible = [pid for pid in policy_ids if result["eligibility"][pid] == "eligible"]
    ranked = result["ranked_policy_ids"]
    grades = label_by_query.get(query_id, {})
    relevant = {pid for pid in eligible if grades.get(pid, 0) >= 2}
    dcg = sum((2 ** grades.get(pid, 0) - 1) / math.log2(rank + 1) for rank, pid in enumerate(ranked[:4], 1))
    ideal = sorted((grades.get(pid, 0) for pid in eligible), reverse=True)[:4]
    idcg = sum((2 ** grade - 1) / math.log2(rank + 1) for rank, grade in enumerate(ideal, 1))
    reciprocal_rank = next((1 / rank for rank, pid in enumerate(ranked, 1) if pid in relevant), 0.0)
    metrics = {
        "p_at_1": float(bool(ranked and ranked[0] in relevant)),
        "p_at_3": len(set(ranked[:3]) & relevant) / 3,
        "recall_at_5": len(set(ranked[:5]) & relevant) / len(relevant) if relevant else 0.0,
        "mrr": reciprocal_rank,
        "ndcg_at_4": dcg / idcg if idcg else 0.0,
        "eligible_relevant_judgments": len(relevant),
        "eligibility_violations": sum(result["eligibility"][pid] != "eligible" for pid in ranked),
        "judged_recommendations": sum(pid in grades for pid in ranked),
        "recommendation_count": len(ranked),
        "tied_rank_fraction": result["tied_rank_fraction"],
        "all_equal_final_scores": result["all_equal_final_scores"],
    }
    return metrics, result
def kg_only_ndcg(query_ids_for_split, weights):
    per_group = {}
    for query_id in query_ids_for_split:
        intent = query_intents[query_id]
        eligible = [pid for pid in policy_ids if eligibility_status(policy_by_id[pid], intent) == "eligible"]
        ranked = sorted(eligible, key=lambda pid: (-kg_preference(policy_by_id[pid], intent, weights), pid))[:4]
        per_group.setdefault(query_groups[query_id], []).append(ndcg_at_4(ranked, query_id, eligible))
    return {group: float(np.mean(scores)) for group, scores in per_group.items()}

query_groups = dict(zip(queries_df["query_id"], queries_df["group_id"]))
train_query_ids = splits["group_split"]["query_ids"]["train"]
validation_query_ids = splits["group_split"]["query_ids"]["validation"]
test_query_ids = splits["group_split"]["query_ids"]["test"]

kg_weight_sweep = []
for weights in kg_grid:
    group_scores = kg_only_ndcg(train_query_ids, weights)
    kg_weight_sweep.append({"weights": weights, "mean_ndcg_at_4": float(np.mean(list(group_scores.values()))) if group_scores else 0.0})
selected_kg_weights = max(
    kg_weight_sweep,
    key=lambda row: (row["mean_ndcg_at_4"], -sum(abs(row["weights"][key] - 1 / 3) for key in ("goal", "risk", "premium"))),
)["weights"]

def profile_scores(query_ids_for_split, model_key, alpha, beta, weights):
    by_group = {}
    for query_id in query_ids_for_split:
        metrics, _ = query_metrics(query_id, model_key, alpha, beta, weights)
        by_group.setdefault(query_groups[query_id], []).append(metrics["ndcg_at_4"])
    return {group: float(np.mean(values)) for group, values in by_group.items()}

def bootstrap_ci(group_scores, seed):
    values = np.asarray(list(group_scores.values()), dtype=float)
    if not len(values):
        return {"low": 0.0, "high": 0.0}
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(1000, len(values)), replace=True).mean(axis=1)
    return {"low": float(np.percentile(samples, 2.5)), "high": float(np.percentile(samples, 97.5))}

def paired_sign_flip_p(candidate, best):
    common = sorted(set(candidate) & set(best))
    differences = np.asarray([candidate[group] - best[group] for group in common], dtype=float)
    if not len(differences):
        return 1.0
    observed = abs(float(differences.mean()))
    sign_patterns = np.asarray(list(itertools.product((-1.0, 1.0), repeat=len(differences))))
    permuted = np.abs((sign_patterns * differences).mean(axis=1))
    return float(np.mean(permuted >= observed - 1e-12))

def holm_adjust(p_values):
    order = np.argsort(p_values)
    adjusted = np.ones(len(p_values), dtype=float)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(p_values) - rank) * float(p_values[index]))
        adjusted[index] = min(1.0, running)
    return adjusted

def sweep_model(model_key):
    rows = []
    for alpha in np.linspace(0, 1, 21):
        for beta in np.linspace(0, 1, 21):
            group_scores = profile_scores(validation_query_ids, model_key, float(alpha), float(beta), selected_kg_weights)
            tie_results = [
                rank_query(query_by_id[qid], query_intents[qid], float(alpha), float(beta), selected_kg_weights, model_key, qid)
                for qid in validation_query_ids
            ] if beta == 0 else []
            tied_lists = sum(result["all_equal_final_scores"] for result in tie_results)
            scored_lists = sum(bool(result["recommendations"]) for result in tie_results)
            all_tied_beta_zero = beta == 0 and scored_lists > 0 and tied_lists == scored_lists
            rows.append({
                "alpha": float(alpha),
                "beta": float(beta),
                "mean_ndcg_at_4": float(np.mean(list(group_scores.values()))) if group_scores else 0.0,
                "validation_profile_count": len(group_scores),
                "ci_95": bootstrap_ci(group_scores, SEED + int(alpha * 100) * 31 + int(beta * 100)),
                "all_equal_final_score_lists": int(tied_lists),
                "scored_validation_list_count": int(scored_lists),
                "informative": not all_tied_beta_zero,
                "selection_note": "NON_INFORMATIVE_ALL_SCORES_TIED" if all_tied_beta_zero else None,
                "_group_scores": group_scores,
            })
    informative_rows = [row for row in rows if row["informative"]]
    tuned = bool(informative_rows)
    if tuned:
        best = max(informative_rows, key=lambda row: (row["mean_ndcg_at_4"], -abs(row["alpha"] - 0.5) - abs(row["beta"] - 0.5), -row["alpha"], -row["beta"]))
    else:
        default_scores = profile_scores(validation_query_ids, model_key, 0.5, 0.5, selected_kg_weights)
        best = {
            "alpha": 0.5,
            "beta": 0.5,
            "mean_ndcg_at_4": float(np.mean(list(default_scores.values()))) if default_scores else 0.0,
            "ci_95": bootstrap_ci(default_scores, SEED),
            "_group_scores": default_scores,
        }
    p_values = [paired_sign_flip_p(row["_group_scores"], best["_group_scores"]) for row in rows]
    p_adjusted = holm_adjust(p_values)
    for row, p_raw, p_adj in zip(rows, p_values, p_adjusted):
        row["paired_sign_flip_p"] = p_raw
        row["holm_adjusted_p"] = float(p_adj)
        row["statistically_indistinguishable_from_best"] = bool(p_adj >= 0.05)
        row.pop("_group_scores")
    return {
        "selected_alpha": best["alpha"],
        "selected_beta": best["beta"],
        "selection_method": "SYNTHETIC_VALIDATION_SWEEP" if tuned else "UNTUNED_DOCUMENTED_DEFAULT",
        "selected_validation_ndcg_at_4": best["mean_ndcg_at_4"],
        "selected_validation_ci_95": best["ci_95"],
        "default_alpha": 0.5,
        "default_beta": 0.5,
        "statistically_indistinguishable_alpha_beta": [
            {"alpha": row["alpha"], "beta": row["beta"], "holm_adjusted_p": row["holm_adjusted_p"]}
            for row in rows if row["statistically_indistinguishable_from_best"]
        ],
        "sweep": rows,
    }
model_sweeps = {key: sweep_model(key) for key in ("baseline", "fine_tuned")}
selected_weights = {
    key: {"alpha": value["selected_alpha"], "beta": value["selected_beta"], "kg_preference_weights": selected_kg_weights}
    for key, value in model_sweeps.items()
}

def summarize_test(model_key, weights, ranker="pipeline"):
    per_query = {}
    rank_results = []
    for query_id in test_query_ids:
        metrics, result = query_metrics(query_id, model_key, weights["alpha"], weights["beta"], weights["kg_preference_weights"], ranker)
        per_query[query_id] = metrics
        rank_results.append(result)
    metric_names = ("p_at_1", "p_at_3", "recall_at_5", "mrr", "ndcg_at_4", "eligibility_violations", "tied_rank_fraction")
    means = {name: float(np.mean([row[name] for row in per_query.values()])) if per_query else 0.0 for name in metric_names}
    tie_metrics = summarize_rank_ties(rank_results)
    return {
        "n_queries": len(per_query),
        "eligible_relevant_judgments": int(sum(row["eligible_relevant_judgments"] for row in per_query.values())),
        "test_customer_profiles": sorted({query_groups[qid] for qid in test_query_ids}),
        "mean_metrics": means,
        "tied_rank_fraction": tie_metrics["tied_rank_fraction"],
        "all_equal_score_list_count": tie_metrics["all_equal_score_list_count"],
        "all_equal_list_fraction": tie_metrics["all_equal_list_fraction"],
        "per_query": per_query,
    }

test_comparison = {key: summarize_test(key, selected_weights[key]) for key in ("baseline", "fine_tuned")}
selected_fine_tuned = selected_weights["fine_tuned"]
ranker_test_results = {
    "pipeline": summarize_test("fine_tuned", selected_fine_tuned),
    "id_order": summarize_test("fine_tuned", selected_fine_tuned, "id_order"),
    "bm25_only": summarize_test("fine_tuned", selected_fine_tuned, "bm25_only"),
}
smoke_records = []
smoke_rank_results = []
violations = 0
returned = 0
for query_id in query_ids:
    metrics, result = query_metrics(query_id, "fine_tuned", selected_weights["fine_tuned"]["alpha"], selected_weights["fine_tuned"]["beta"], selected_kg_weights)
    returned += len(result["recommendations"])
    violations += sum(result["eligibility"][rec["policy_id"]] != "eligible" for rec in result["recommendations"])
    smoke_rank_results.append(result)
    smoke_records.append({"query_id": query_id, "customer_profile": query_groups[query_id], "status": result["status"], "recommendations": result["recommendations"], "eligible_candidate_count": result["eligible_candidate_count"], "display_cap": result["display_cap"], "display_omitted_count": result["display_omitted_count"], "tied_rank_fraction": result["tied_rank_fraction"], "all_equal_final_scores": result["all_equal_final_scores"]})
smoke_tie_metrics = assert_tie_collapse_limit(smoke_rank_results)
retrieval_smoke = {
    "mode": "SYNTHETIC_SMOKE_TEST",
    "provenance": "SYNTHETIC",
    "status": "COMPLETED_SYNTHETIC_SMOKE_TEST",
    "query_count": int(len(smoke_records)),
    "recommendation_count": int(returned),
    "candidate_cap_effect": {"display_cap": int(CONFIG["display_recommendations"]), "eligible_candidates_ranked": int(sum(row["eligible_candidate_count"] for row in smoke_records)), "eligible_candidates_omitted_from_display": int(sum(row["display_omitted_count"] for row in smoke_records))},
    "eligibility_violations": int(violations),
    "eligibility_violation_rate": float(violations / returned) if returned else 0.0,
    "returned_recommendations_are_eligible": violations == 0,
    "tied_rank_fraction": smoke_tie_metrics["tied_rank_fraction"],
    "all_equal_score_list_count": smoke_tie_metrics["all_equal_score_list_count"],
    "all_equal_list_fraction": smoke_tie_metrics["all_equal_list_fraction"],
    "backend": BACKENDS,
    "cross_encoder_fallback_reason": cross_encoder_error,
    "human_labels_used": False,
    "latency_measured": False,
    "reason": "Synthetic smoke labels validate pipeline wiring only; these results are not evidence of real-world relevance or KG benefit.",
    "queries": smoke_records,
}
weights_smoke = {
    "mode": "SYNTHETIC_SMOKE_TEST",
    "provenance": "SYNTHETIC",
    "status": "COMPLETED_SYNTHETIC_SMOKE_TEST",
    "human_labels_used": False,
    "validation_customer_profiles": sorted({query_groups[qid] for qid in validation_query_ids}),
    "selected_kg_preference_weights": selected_kg_weights,
    "kg_preference_training_sweep": kg_weight_sweep,
    "models": model_sweeps,
    "selected_alpha": selected_weights["fine_tuned"]["alpha"],
    "selected_beta": selected_weights["fine_tuned"]["beta"],
    "sweep": model_sweeps["fine_tuned"]["sweep"],
    "selection_method": model_sweeps["fine_tuned"]["selection_method"],
    "default_weights": {"alpha": 0.5, "beta": 0.5, "tuned": False},
    "limitation": "Synthetic results are not evidence of real-world relevance or KG benefit.",
}
evaluation = {
    "mode": "SYNTHETIC_SMOKE_TEST",
    "provenance": "SYNTHETIC",
    "human_labels_used": False,
    "train_customer_profiles": sorted(splits["group_split"]["groups"]["train"]),
    "validation_customer_profiles": sorted(splits["group_split"]["groups"]["validation"]),
    "test_customer_profiles": sorted(splits["group_split"]["groups"]["test"]),
    "test_metric_status": "EVALUABLE" if test_comparison["fine_tuned"]["eligible_relevant_judgments"] else "NOT_EVALUABLE_NO_ELIGIBLE_RELEVANT_JUDGMENTS",
    "test_results": test_comparison,
    "ranker_results": ranker_test_results,
    "baselines": {"id_order": ranker_test_results["id_order"], "bm25_only": ranker_test_results["bm25_only"]},
    "comparison": {
        metric: test_comparison["fine_tuned"]["mean_metrics"][metric] - test_comparison["baseline"]["mean_metrics"][metric]
        for metric in ("p_at_1", "p_at_3", "recall_at_5", "mrr", "ndcg_at_4")
    },
    "eligibility_violations": int(violations),
    "limitation": "A small synthetic label set is used only to exercise fine-tuning and evaluation code; no real-world relevance or KG benefit is claimed.",
}
(RESULTS / "retrieval_smoke.json").write_text(json.dumps(retrieval_smoke, indent=2), encoding="utf-8")
(RESULTS / "weights_smoke.json").write_text(json.dumps(weights_smoke, indent=2), encoding="utf-8")
(RESULTS / "synthetic_smoke_evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")
print(json.dumps({key: value for key, value in retrieval_smoke.items() if key != "queries"}, indent=2))
print(json.dumps({"status": weights_smoke["status"], "selected_alpha": weights_smoke["selected_alpha"], "selected_beta": weights_smoke["selected_beta"], "kg_preference_weights": selected_kg_weights}, indent=2))
print(json.dumps({key: value["mean_metrics"] for key, value in test_comparison.items()}, indent=2))



{
  "mode": "SYNTHETIC_SMOKE_TEST",
  "provenance": "SYNTHETIC",
  "status": "COMPLETED_SYNTHETIC_SMOKE_TEST",
  "query_count": 18,
  "recommendation_count": 74,
  "eligibility_violations": 0,
  "eligibility_violation_rate": 0.0,
  "returned_recommendations_are_eligible": true,
  "backend": {
    "vector": "faiss_exact_ip",
    "encoder": "pretrained_biencoder",
    "cross_encoder": "cross_encoder_ranking_score",
    "network": "allowed"
  },
  "cross_encoder_fallback_reason": null,
  "human_labels_used": false,
  "latency_measured": false,
  "reason": "Synthetic smoke labels validate pipeline wiring only; these results are not evidence of real-world relevance or KG benefit."
}
{
  "status": "COMPLETED_SYNTHETIC_SMOKE_TEST",
  "selected_alpha": 0.5,
  "selected_beta": 0.5,
  "kg_preference_weights": {
    "goal": 0.25,
    "risk": 0.25,
    "premium": 0.5
  }
}
{
  "baseline": {
    "p_at_1": 0.0,
    "p_at_3": 0.0,
    "recall_at_5": 0.0,
    "mrr": 0.0,
    "ndcg_at_4": 0.0,
    "el

## Assertions

In [3]:
assert weights_smoke["status"] == "COMPLETED_SYNTHETIC_SMOKE_TEST"
assert 0 <= weights_smoke["selected_alpha"] <= 1 and 0 <= weights_smoke["selected_beta"] <= 1
assert len(weights_smoke["sweep"]) == 441
assert abs(sum(weights_smoke["selected_kg_preference_weights"].values()) - 1) < 1e-12
assert label_status["human_graded_rows"] == 0
assert weights_smoke["human_labels_used"] is False
assert evaluation["provenance"] == "SYNTHETIC"
assert evaluation["test_customer_profiles"] == sorted(splits["group_split"]["groups"]["test"])
assert not (set(evaluation["train_customer_profiles"]) & set(evaluation["validation_customer_profiles"]))
assert not (set(evaluation["train_customer_profiles"]) & set(evaluation["test_customer_profiles"]))
assert retrieval_smoke["query_count"] == len(queries_df)
assert retrieval_smoke["eligibility_violations"] == 0
assert retrieval_smoke["eligibility_violation_rate"] == 0.0
assert all(rec["score_kind"] == "ranking_score_not_probability" for row in smoke_records for rec in row["recommendations"])
assert all(rec["eligibility"] == "eligible" for row in smoke_records for rec in row["recommendations"])

old = rank_query("age 95", {"customer_age": 95, "term": None, "all_goals": []})
assert old["status"] == "NO_ELIGIBLE_POLICIES" and not old["recommendations"]

# A constant score shift must not change rank order.
base = {"a": 0.1, "b": 0.7, "c": 0.4}
shifted = {key: value + 11 for key, value in base.items()}
assert sorted(base, key=lambda key: (-base[key], key)) == sorted(shifted, key=lambda key: (-shifted[key], key))

fallback = backend_plan(False, False, False, False)
assert fallback == {"vector": "numpy_exact", "encoder": "tfidf_fallback", "cross_encoder": "dense_ranking_score_fallback", "network": "disabled_local_only"}
without_faiss = dense_scores("life cover", [policy_ids[0], policy_ids[1]], use_faiss=False)
assert len(without_faiss) == 2 and np.isfinite(without_faiss).all()
assert minmax([1.0, 1.0]).tolist() == [0.5, 0.5]

for query_row in queries_df.itertuples(index=False):
    first = rank_query(query_row.raw_query, intents[query_row.query_id]["intent"])
    second = rank_query(query_row.raw_query, intents[query_row.query_id]["intent"])
    assert [r["policy_id"] for r in first["recommendations"]] == [r["policy_id"] for r in second["recommendations"]]
    scores = [r["final_ranking_score"] for r in first["recommendations"]]
    assert all(a >= b for a, b in zip(scores, scores[1:]))

print("Notebook 04 checks passed: synthetic-only sweep, held-out profile evaluation, strict eligibility, and deterministic rankings.")

Notebook 04 checks passed: synthetic-only sweep, held-out profile evaluation, strict eligibility, and deterministic rankings.


In [ ]:
human_eval_queries = pd.read_csv(ARTIFACTS / "human_eval_queries.csv", dtype=str).fillna("")
human_eval_labels = pd.read_parquet(ARTIFACTS / "human_eval_labels.parquet")
human_eval_intents = {row["query_id"]: row["intent"] for row in json.loads((ARTIFACTS / "human_eval_intents.json").read_text(encoding="utf-8"))}
human_ids = human_eval_queries["query_id"].tolist()
if set(human_ids) & set(query_ids):
    raise ValueError("Human evaluation queries must not overlap synthetic smoke queries")

human_encoder = SentenceTransformer(model_paths["fine_tuned"], device=DEVICE)
human_vectors = human_encoder.encode(human_eval_queries["raw_query"].tolist(), batch_size=16, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
del human_encoder
semantic_vectors["fine_tuned"]["queries"].update(dict(zip(human_ids, human_vectors)))
query_position.update({query_id: len(query_position) + index for index, query_id in enumerate(human_ids)})

human_pairs = []
for query_id in human_ids:
    human_pairs.extend((query_id, policy_id) for policy_id in policy_ids if eligibility_status(policy_by_id[policy_id], human_eval_intents[query_id]) == "eligible")
if cross_encoder is not None and human_pairs:
    human_scores = cross_encoder.predict(
        [[human_eval_queries.iloc[human_ids.index(query_id)]["raw_query"], documents[policy_id]] for query_id, policy_id in human_pairs],
        batch_size=16, show_progress_bar=False, convert_to_numpy=True,
    )
    for (query_id, policy_id), score in zip(human_pairs, np.asarray(human_scores).reshape(-1)):
        cross_scores.setdefault(query_id, {})[policy_id] = float(score)

human_labels_by_query = {
    query_id: dict(zip(rows["policy_id"], rows["grade"].astype(int)))
    for query_id, rows in human_eval_labels.groupby("query_id", sort=False)
}
human_candidate_rows = []
human_per_query = {}
human_violations = 0
human_pipeline_results = []
for query in human_eval_queries.itertuples(index=False):
    query_id = query.query_id
    result = rank_query(
        query.raw_query, human_eval_intents[query_id],
        selected_weights["fine_tuned"]["alpha"], selected_weights["fine_tuned"]["beta"],
        selected_kg_weights, "fine_tuned", query_id,
    )
    human_pipeline_results.append(result)
    human_violations += sum(result["eligibility"].get(pid) != "eligible" for pid in result["ranked_policy_ids"])
    grades = human_labels_by_query.get(query_id, {})
    eligible_judged = {pid: grade for pid, grade in grades.items() if result["eligibility"].get(pid) == "eligible"}
    ranked_judged = [pid for pid in result["ranked_policy_ids"] if pid in eligible_judged]
    relevant = {pid for pid, grade in eligible_judged.items() if grade >= 2}
    evaluable = bool(relevant)
    dcg = sum((2 ** eligible_judged[pid] - 1) / math.log2(rank + 1) for rank, pid in enumerate(ranked_judged[:4], 1))
    ideal = sorted(eligible_judged.values(), reverse=True)[:4]
    idcg = sum((2 ** grade - 1) / math.log2(rank + 1) for rank, grade in enumerate(ideal, 1))
    metrics = {
        "p_at_1": float(bool(ranked_judged and ranked_judged[0] in relevant)),
        "p_at_3": sum(pid in relevant for pid in ranked_judged[:3]) / 3,
        "recall_at_5": len(set(ranked_judged[:5]) & relevant) / len(relevant) if relevant else 0.0,
        "mrr": next((1 / rank for rank, pid in enumerate(ranked_judged, 1) if pid in relevant), 0.0),
        "ndcg_at_4": dcg / idcg if idcg else 0.0,
        "tied_rank_fraction": result["tied_rank_fraction"],
    }
    human_per_query[query_id] = {
        "group_id": query.group_id,
        "status": "EVALUABLE" if evaluable else "NO_ELIGIBLE_RELEVANT_JUDGMENTS",
        "judged_policy_count": len(grades),
        "eligible_judged_policy_count": len(eligible_judged),
        "eligible_relevant_judgment_count": len(relevant),
        "returned_judged_policy_count": len(ranked_judged),
        "metrics": metrics,
    }
    for record in result["recommendations"]:
        pid = record["policy_id"]
        human_candidate_rows.append({
            "query_id": query_id, "group_id": query.group_id, "raw_query": query.raw_query,
            "query_provenance": query.provenance, "rank": record["rank"], "policy_id": pid,
            "policy_name": policy_by_id[pid]["policy_name"], "human_grade": grades.get(pid, ""),
            "eligibility": record["eligibility"], "final_ranking_score": record["final_ranking_score"],
            "score_kind": record["score_kind"],
        })

metric_names = ("p_at_1", "p_at_3", "recall_at_5", "mrr", "ndcg_at_4")
evaluable_queries = {qid: row for qid, row in human_per_query.items() if row["status"] == "EVALUABLE"}
group_metrics = {}
for group_id in sorted({row["group_id"] for row in evaluable_queries.values()}):
    rows = [row["metrics"] for row in evaluable_queries.values() if row["group_id"] == group_id]
    group_metrics[group_id] = {name: float(np.mean([row[name] for row in rows])) for name in metric_names}
human_pipeline_ties = assert_tie_collapse_limit(human_pipeline_results)


def summarize_human_baseline(ranker):
    per_query = {}
    rank_results = []
    for query in human_eval_queries.itertuples(index=False):
        query_id = query.query_id
        result = rank_query(
            query.raw_query, human_eval_intents[query_id],
            selected_weights["fine_tuned"]["alpha"], selected_weights["fine_tuned"]["beta"],
            selected_kg_weights, "fine_tuned", query_id, ranker,
        )
        rank_results.append(result)
        grades = human_labels_by_query.get(query_id, {})
        eligible_judged = {pid: grade for pid, grade in grades.items() if result["eligibility"].get(pid) == "eligible"}
        ranked_judged = [pid for pid in result["ranked_policy_ids"] if pid in eligible_judged]
        relevant = {pid for pid, grade in eligible_judged.items() if grade >= 2}
        dcg = sum((2 ** eligible_judged[pid] - 1) / math.log2(rank + 1) for rank, pid in enumerate(ranked_judged[:4], 1))
        ideal = sorted(eligible_judged.values(), reverse=True)[:4]
        idcg = sum((2 ** grade - 1) / math.log2(rank + 1) for rank, grade in enumerate(ideal, 1))
        per_query[query_id] = {
            "group_id": query.group_id,
            "evaluable": bool(relevant),
            "metrics": {
                "p_at_1": float(bool(ranked_judged and ranked_judged[0] in relevant)),
                "p_at_3": sum(pid in relevant for pid in ranked_judged[:3]) / 3,
                "recall_at_5": len(set(ranked_judged[:5]) & relevant) / len(relevant) if relevant else 0.0,
                "mrr": next((1 / rank for rank, pid in enumerate(ranked_judged, 1) if pid in relevant), 0.0),
                "ndcg_at_4": dcg / idcg if idcg else 0.0,
                "tied_rank_fraction": result["tied_rank_fraction"],
            },
        }
    evaluable = {qid: row for qid, row in per_query.items() if row["evaluable"]}
    group_results = {}
    for group_id in sorted({row["group_id"] for row in evaluable.values()}):
        rows = [row["metrics"] for row in evaluable.values() if row["group_id"] == group_id]
        group_results[group_id] = {name: float(np.mean([row[name] for row in rows])) for name in metric_names}
    means = {name: float(np.mean([row[name] for row in group_results.values()])) if group_results else 0.0 for name in metric_names}
    return {
        "n_queries": len(per_query),
        "evaluable_query_count": len(evaluable),
        "mean_metrics": means,
        "candidate_cap_effect": {"display_cap": int(CONFIG["display_recommendations"]), "eligible_candidates_ranked": int(sum(result["eligible_candidate_count"] for result in rank_results)), "eligible_candidates_omitted_from_display": int(sum(result["display_omitted_count"] for result in rank_results))},
        **summarize_rank_ties(rank_results),
    }


human_ranker_results = {
    "pipeline": {
        "n_queries": len(human_eval_queries),
        "evaluable_query_count": len(evaluable_queries),
        "mean_metrics": group_metrics,
        "candidate_cap_effect": {"display_cap": int(CONFIG["display_recommendations"]), "eligible_candidates_ranked": int(sum(result["eligible_candidate_count"] for result in human_pipeline_results)), "eligible_candidates_omitted_from_display": int(sum(result["display_omitted_count"] for result in human_pipeline_results))},
        **human_pipeline_ties,
    },
    "id_order": summarize_human_baseline("id_order"),
    "bm25_only": summarize_human_baseline("bm25_only"),
}
human_evaluation = {
    "mode": "HUMAN_REVIEWED_EVALUATION",
    "provenance": "HUMAN_REVIEWED_BY_USER",
    "query_provenance": sorted(human_eval_queries["provenance"].unique().tolist()),
    "query_count": int(len(human_eval_queries)),
    "graded_query_count": int(len(human_labels_by_query)),
    "ungraded_query_ids": sorted(set(human_ids) - set(human_labels_by_query)),
    "graded_rows": int(len(human_eval_labels)),
    "evaluable_query_count": int(len(evaluable_queries)),
    "non_evaluable_query_ids": sorted(set(human_ids) - set(evaluable_queries)),
    "eligibility_violations": int(human_violations),
    "tied_rank_fraction": human_pipeline_ties["tied_rank_fraction"],
    "candidate_cap_effect": {"display_cap": int(CONFIG["display_recommendations"]), "eligible_candidates_ranked": int(sum(result["eligible_candidate_count"] for result in human_pipeline_results)), "eligible_candidates_omitted_from_display": int(sum(result["display_omitted_count"] for result in human_pipeline_results))},
    "ranker_results": human_ranker_results,
    "baselines": {"id_order": human_ranker_results["id_order"], "bm25_only": human_ranker_results["bm25_only"]},
    "metrics_status": "EVALUABLE" if evaluable_queries else "NOT_EVALUABLE_NO_ELIGIBLE_RELEVANT_JUDGMENTS",
    "metrics_scope": "Macro-averaged by query group; ranked eligible judgments only; unjudged policies excluded.",
    "training_labels_used": False,
    "human_labels_used_for_tuning": False,
    "weights_source": "selected on synthetic validation data; not tuned on human evaluation labels",
    "group_macro_metrics": group_metrics,
    "per_query": human_per_query,
}
(RESULTS / "human_evaluation_candidates.csv").write_text(pd.DataFrame(human_candidate_rows).to_csv(index=False), encoding="utf-8")
(RESULTS / "human_evaluation.json").write_text(json.dumps(human_evaluation, indent=2), encoding="utf-8")
assert human_violations == 0
assert human_evaluation["training_labels_used"] is False and human_evaluation["human_labels_used_for_tuning"] is False
assert all(record["eligibility"] == "eligible" for record in human_candidate_rows)
print(f"Human evaluation complete: {len(evaluable_queries)} evaluable queries; results kept separate from synthetic smoke metrics.")